In [1]:
# https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#streamable-http

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langchain_mcp_adapters.tools import load_mcp_tools # uv add langchain-mcp-adapters

async with streamablehttp_client("http://127.0.0.1:8000/mcp/") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # 'hello' 도구를 비동기적으로 실행하고 결과를 result 변수에 할당
        # MCP는 서버이기 때문에 네트워크로 비동기 통신함
        result = await tools[0].ainvoke({"name": "김일남"})
        print(result)

[{'type': 'text', 'text': 'Hello, 김일남!', 'id': 'lc_7540d409-86d5-4f54-8a95-b1b1f6db415b'}]


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tools": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='greet', args_schema={'additionalProperties': False, 'properties': {'name': {'type': 'string'}}, 'required': ['name'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000018AA75B2CA0>),
 StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수\n\nArgs:\n    timezone: 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n    location: 지역명, 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨", args_schema={'additionalProperties': False, 'properties': {'timezone': {'type': 'string'}, 'location': {'type': 'string'}}, 'required': ['timezone', 'location'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000018AA765AFC0>),
 StructuredTool(name='get_yf_stock_history', description='주식 종목의 가격 데이터를 조회하는 함수', args_sche

In [4]:
result = await tools[1].ainvoke({"timezone": "Asia/Seoul", "location": "부산"})
result

[{'type': 'text',
  'text': 'Asia/Seoul (부산) 현재시각 2026-04-11 15:49:09',
  'id': 'lc_db2bbe7e-4cd4-4ea2-8965-30fc07a6a2b8'}]